# WorldSim DGX Spark QLoRA Smoke Run

Thin notebook wrapper around the shared `training.lib.qlora_smoke` API.


In [1]:
!uv pip install datasets transformers peft accelerate safetensors sentencepiece

Using Python 3.12.3 environment at: /home/hyunlord/github/worldsim-training/.venv
Audited 6 packages in 2ms


In [4]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device0:", torch.cuda.get_device_name(0))

torch: 2.12.0.dev20260225+cu130
cuda: 13.0
cuda_available: True
device_count: 1
device0: NVIDIA GB10


In [5]:
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.49.2


In [6]:
import sys                                                                                                                                                                                                               
from pathlib import Path                                                                                                                                                                                               
                                                                                                                                                                                                                       
# 프로젝트 루트(notebooks의 상위)를 경로에 추가                                                                                                                                                                          
project_root = Path.cwd().parent                                                                                                                                                                                         
sys.path.insert(0, str(project_root))    

In [7]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
environment


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': {'version': '3.12.3',
  'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'},
 'platform': {'system': 'Linux',
  'release': '6.14.0-1015-nvidia',
  'machine': 'aarch64'},
 'cwd': '/home/hyunlord/github/worldsim-training/notebooks',
 'torch': {'available': True,
  'version': '2.12.0.dev20260225+cu130',
  'cuda_available': True,
  'mps_available': False,
  'cuda_device_count': 1,
  'cuda_device_name': 'NVIDIA GB10',
  'cuda_bf16_supported': True},
 'transformers': {'available': True, 'version': '5.2.0'},
 'datasets': {'available': True, 'version': '4.7.0'},
 'peft': {'available': True, 'version': '0.18.1'},
 'accelerate': {'available': True, 'version': '1.12.0'},
 'bitsandbytes': {'available': True, 'version': '0.49.2'}}

In [8]:
from training.lib.qlora_smoke import get_environment_summary

preflight = get_environment_summary()
torch_info = preflight.get("torch", {})
bnb_info = preflight.get("bitsandbytes", {})

assert torch_info.get("cuda_available"), "CUDA is unavailable in this Jupyter kernel. Run this notebook on a DGX Spark CUDA kernel."
assert bnb_info.get("available"), "bitsandbytes is unavailable. Install requirements-training.txt in the notebook environment."
preflight


{'python': {'version': '3.12.3',
  'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'},
 'platform': {'system': 'Linux',
  'release': '6.14.0-1015-nvidia',
  'machine': 'aarch64'},
 'cwd': '/home/hyunlord/github/worldsim-training/notebooks',
 'torch': {'available': True,
  'version': '2.12.0.dev20260225+cu130',
  'cuda_available': True,
  'mps_available': False,
  'cuda_device_count': 1,
  'cuda_device_name': 'NVIDIA GB10',
  'cuda_bf16_supported': True},
 'transformers': {'available': True, 'version': '5.2.0'},
 'datasets': {'available': True, 'version': '4.7.0'},
 'peft': {'available': True, 'version': '0.18.1'},
 'accelerate': {'available': True, 'version': '1.12.0'},
 'bitsandbytes': {'available': True, 'version': '0.49.2'}}

In [9]:
from datetime import UTC, datetime
from pathlib import Path

from training.lib.qlora_smoke import SmokeRunConfig

run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
config = SmokeRunConfig(
    train_file=Path("data/training/worldsim-v31-mix-v1/train_converted.jsonl"),
    dev_file=Path("data/training/worldsim-v31-mix-v1/dev_converted.jsonl"),
    output_dir=Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1") / run_id,
    max_steps=5,
    max_train_samples=64,
    max_eval_samples=32,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    require_qlora=True,
)
config.to_dict()

{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'train_file': 'data/training/worldsim-v31-mix-v1/train_converted.jsonl',
 'dev_file': 'data/training/worldsim-v31-mix-v1/dev_converted.jsonl',
 'output_dir': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260310T002502Z',
 'max_steps': 5,
 'max_train_samples': 64,
 'max_eval_samples': 32,
 'max_length': 512,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 1,
 'learning_rate': 0.0002,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'seed': 42,
 'trust_remote_code': False,
 'disable_qlora': False,
 'require_qlora': True,
 'dry_run': False}

In [10]:
from training.lib.qlora_smoke import run_smoke_or_raise

result = run_smoke_or_raise(config)
result.to_dict()

Loading weights:   1%|          | 3/290 [00:01<03:18,  1.44it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 290/290 [00:04<00:00, 58.35it/s, Materializing param=model.norm.weight]                               
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


SmokeRunBlockedError: ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
from training.lib.qlora_smoke import preview_metrics

metrics = preview_metrics(result.output_dir)
metrics

In [ ]:
from training.lib.qlora_smoke import count_parseable_json_samples, load_sample_generations

samples = load_sample_generations(result.output_dir)
{
    "sample_count": len(samples),
    **count_parseable_json_samples(samples),
}

In [ ]:
samples[:3]

In [ ]:
{
    "status": result.status,
    "used_true_qlora": result.used_true_qlora,
    "output_dir": result.output_dir,
    "summary_path": result.summary_path,
    "metrics_path": result.metrics_path,
    "sample_path": result.sample_path,
    "adapter_dir": result.adapter_dir,
}